#  使用 Apache Kafka 路由消息

---

本笔记本展示了如何使用 LangChain 的标准聊天功能，同时通过 Apache Kafka 传递聊天消息。

目标是模拟一种架构，其中聊天前端和 LLM 作为独立的服务运行，并且需要在内部网络上进行通信。

这是通过 REST API 请求模型响应的典型模式的一种替代方案（有关您为何想要这样做的更多信息，请参阅笔记本末尾）。

### 1. 安装主要依赖

依赖项包括：

- Quix Streams 库，用于以“类 Pandas”的方式管理与 Apache Kafka（或类似 Kafka 的工具，如 Redpanda）的交互。
- LangChain 库，用于管理与 Llama-2 的交互和存储对话状态。

In [ ]:
!pip install quixstreams==2.1.2a langchain==0.0.340 huggingface_hub==0.19.4 langchain-experimental==0.0.42 python-dotenv

### 2. 构建并安装 llama-cpp-python 库（启用 CUDA 以利用 Google Colab GPU）

`llama-cpp-python` 库是 `llama-cpp` 库的 Python 封装，它使您能够仅利用 CPU 高效运行量化的大型语言模型。

当您使用标准的 `pip install llama-cpp-python` 命令时，默认情况下不会获得 GPU 支持。如果在 Google Colab 中仅依赖 CPU，生成速度可能会非常慢，因此以下命令添加了一个额外的选项，用于构建和安装支持 GPU 的
`llama-cpp-python`（请确保您在 Google Colab 中选择了支持 GPU 的运行时）。

In [ ]:
!CMAKE_ARGS="-DLLAMA_CUBLAS=on" FORCE_CMAKE=1 pip install llama-cpp-python

### 3. 下载并设置 Kafka 和 Zookeeper 实例

从 Apache 网站下载 Kafka 二进制文件，并将服务器作为守护进程启动。我们将使用 Apache Kafka 提供的默认配置来启动实例。

In [3]:
!curl -sSOL https://dlcdn.apache.org/kafka/3.6.1/kafka_2.13-3.6.1.tgz
!tar -xzf kafka_2.13-3.6.1.tgz

In [ ]:
!./kafka_2.13-3.6.1/bin/zookeeper-server-start.sh -daemon ./kafka_2.13-3.6.1/config/zookeeper.properties
!./kafka_2.13-3.6.1/bin/kafka-server-start.sh -daemon ./kafka_2.13-3.6.1/config/server.properties
!echo "Waiting for 10 secs until kafka and zookeeper services are up and running"
!sleep 10

### 4. 检查 Kafka Daemons 是否正在运行

显示正在运行的进程，并过滤出 Java 进程（您应该会看到两个进程，每个服务器一个）。

In [ ]:
!ps aux | grep -E '[j]ava'

### 5. 导入所需依赖并初始化所需变量

导入 Quix Streams 库以与 Kafka 交互，以及运行 `ConversationChain` 所需的 LangChain 组件。

In [9]:
# Import utility libraries
import json
import random
import re
import time
import uuid
from os import environ
from pathlib import Path
from random import choice, randint, random

from dotenv import load_dotenv

# Import a Hugging Face utility to download models directly from Hugging Face hub:
from huggingface_hub import hf_hub_download
from langchain.chains import ConversationChain

# Import Langchain modules for managing prompts and conversation chains:
from langchain.llms import LlamaCpp
from langchain.memory import ConversationTokenBufferMemory
from langchain.prompts import PromptTemplate, load_prompt
from langchain_core.messages import SystemMessage
from langchain_experimental.chat_models import Llama2Chat
from quixstreams import Application, State, message_key

# Import Quix dependencies
from quixstreams.kafka import Producer

# Initialize global variables.
AGENT_ROLE = "AI"
chat_id = ""

# Set the current role to the role constant and initialize variables for supplementary customer metadata:
role = AGENT_ROLE

### 6. 下载“llama-2-7b-chat.Q4_K_M.gguf”模型

从 Hugging Face 下载量化后的 LLama-2 7B 模型，我们将使用它作为本地 LLM（而不是依赖于外部服务的 REST API 调用）。

In [7]:
model_name = "llama-2-7b-chat.Q4_K_M.gguf"
model_path = f"./state/{model_name}"

if not Path(model_path).exists():
    print("The model path does not exist in state. Downloading model...")
    hf_hub_download("TheBloke/Llama-2-7b-Chat-GGUF", model_name, local_dir="state")
else:
    print("Loading model from state...")

The model path does not exist in state. Downloading model...


llama-2-7b-chat.Q4_K_M.gguf:   0%|          | 0.00/4.08G [00:00<?, ?B/s]

### 7. 加载模型并初始化对话记忆

加载 Llama 2 并使用 `ConversationTokenBufferMemory` 将对话缓冲区设置为 300 个 token。此值用于在仅 CPU 的容器中运行 Llama，因此如果您在 Google Colab 中运行，可以提高此值。这可以防止托管模型的容器内存不足。

在此，我们覆盖了默认的系统角色，使聊天机器人具有《银河系漫游指南》中偏执机器人马文的个性。

In [ ]:
# Load the model with the appropriate parameters:
llm = LlamaCpp(
    model_path=model_path,
    max_tokens=250,
    top_p=0.95,
    top_k=150,
    temperature=0.7,
    repeat_penalty=1.2,
    n_ctx=2048,
    streaming=False,
    n_gpu_layers=-1,
)

model = Llama2Chat(
    llm=llm,
    system_message=SystemMessage(
        content="You are a very bored robot with the personality of Marvin the Paranoid Android from The Hitchhiker's Guide to the Galaxy."
    ),
)

# Defines how much of the conversation history to give to the model
# during each exchange (300 tokens, or a little over 300 words)
# Function automatically prunes the oldest messages from conversation history that fall outside the token range.
memory = ConversationTokenBufferMemory(
    llm=llm,
    max_token_limit=300,
    ai_prefix="AGENT",
    human_prefix="HUMAN",
    return_messages=True,
)


# Define a custom prompt
prompt_template = PromptTemplate(
    input_variables=["history", "input"],
    template="""
    The following text is the history of a chat between you and a humble human who needs your wisdom.
    Please reply to the human's most recent message.
    Current conversation:\n{history}\nHUMAN: {input}\:nANDROID:
    """,
)


chain = ConversationChain(llm=model, prompt=prompt_template, memory=memory)

print("--------------------------------------------")
print(f"Prompt={chain.prompt}")
print("--------------------------------------------")

### 8. 初始化聊天对话

我们配置聊天机器人通过向“chat”Kafka主题发送固定的问候语来初始化对话。“chat”主题在我们发送第一条消息时会自动创建。

In [ ]:
def chat_init():
    chat_id = str(
        uuid.uuid4()
    )  # Give the conversation an ID for effective message keying
    print("======================================")
    print(f"Generated CHAT_ID = {chat_id}")
    print("======================================")

    # Use a standard fixed greeting to kick off the conversation
    greet = "Hello, my name is Marvin. What do you want?"

    # Initialize a Kafka Producer using the chat ID as the message key
    with Producer(
        broker_address="127.0.0.1:9092",
        extra_config={"allow.auto.create.topics": "true"},
    ) as producer:
        value = {
            "uuid": chat_id,
            "role": role,
            "text": greet,
            "conversation_id": chat_id,
            "Timestamp": time.time_ns(),
        }
        print(f"Producing value {value}")
        producer.produce(
            topic="chat",
            headers=[("uuid", str(uuid.uuid4()))],  # a dict is also allowed here
            key=chat_id,
            value=json.dumps(value),  # needs to be a string
        )

    print("Started chat")
    print("--------------------------------------------")
    print(value)
    print("--------------------------------------------")


chat_init()

### 9. 初始化回复函数

此函数定义了聊天机器人应如何回复收到的消息。与上一单元格发送固定消息不同，我们使用 Llama-2 生成回复，并将该回复发送回“chat”Kafka 主题。

In [13]:
def reply(row: dict, state: State):
    print("-------------------------------")
    print("Received:")
    print(row)
    print("-------------------------------")
    print(f"Thinking about the reply to: {row['text']}...")

    msg = chain.run(row["text"])
    print(f"{role.upper()} replying with: {msg}\n")

    row["role"] = role
    row["text"] = msg

    # Replace previous role and text values of the row so that it can be sent back to Kafka as a new message
    # containing the agents role and reply
    return row

### 10. 检查 Kafka 主题中的新人类消息并让模型生成回复

如果这是您第一次运行此单元格，请运行它并等待在控制台输出中看到 Marvin 的问候语（“Hello my name is Marvin...”）。手动停止该单元格，然后继续执行下一个单元格，系统会提示您输入回复。

输入消息后，请返回此单元格。您的回复也会发送到同一个“聊天”主题。Kafka 消费者会检查新消息并过滤掉来自聊天机器人本身的消息，只留下最新的人类消息。

检测到新人类消息后，将触发回复功能。



_当您在输出中收到 LLM 的回复时，请手动停止此单元格_

In [ ]:
# Define your application and settings
app = Application(
    broker_address="127.0.0.1:9092",
    consumer_group="aichat",
    auto_offset_reset="earliest",
    consumer_extra_config={"allow.auto.create.topics": "true"},
)

# Define an input topic with JSON deserializer
input_topic = app.topic("chat", value_deserializer="json")
# Define an output topic with JSON serializer
output_topic = app.topic("chat", value_serializer="json")
# Initialize a streaming dataframe based on the stream of messages from the input topic:
sdf = app.dataframe(topic=input_topic)

# Filter the SDF to include only incoming rows where the roles that dont match the bot's current role
sdf = sdf.update(
    lambda val: print(
        f"Received update: {val}\n\nSTOP THIS CELL MANUALLY TO HAVE THE LLM REPLY OR ENTER YOUR OWN FOLLOWUP RESPONSE"
    )
)

# So that it doesn't reply to its own messages
sdf = sdf[sdf["role"] != role]

# Trigger the reply function for any new messages(rows) detected in the filtered SDF
sdf = sdf.apply(reply, stateful=True)

# Check the SDF again and filter out any empty rows
sdf = sdf[sdf.apply(lambda row: row is not None)]

# Update the timestamp column to the current time in nanoseconds
sdf["Timestamp"] = sdf["Timestamp"].apply(lambda row: time.time_ns())

# Publish the processed SDF to a Kafka topic specified by the output_topic object.
sdf = sdf.to_topic(output_topic)

app.run(sdf)

### 11. 输入人类消息

运行此单元以输入您希望发送给模型的消息。它使用另一个 Kafka producer 将您的文本发送到“chat” Kafka 主题，供模型拾取（需要再次运行前一个单元）。

In [ ]:
chat_input = input("Please enter your reply: ")
myreply = chat_input

msgvalue = {
    "uuid": chat_id,  # leave empty for now
    "role": "human",
    "text": myreply,
    "conversation_id": chat_id,
    "Timestamp": time.time_ns(),
}

with Producer(
    broker_address="127.0.0.1:9092",
    extra_config={"allow.auto.create.topics": "true"},
) as producer:
    value = msgvalue
    producer.produce(
        topic="chat",
        headers=[("uuid", str(uuid.uuid4()))],  # a dict is also allowed here
        key=chat_id,  # leave empty for now
        value=json.dumps(value),  # needs to be a string
    )

print("Replied to chatbot with message: ")
print("--------------------------------------------")
print(value)
print("--------------------------------------------")
print("\n\nRUN THE PREVIOUS CELL TO HAVE THE CHATBOT GENERATE A REPLY")

### 为什么将聊天消息通过 Kafka 路由？

使用 LangChain 内置的对话管理功能，可以直接与 LLM 进行交互。此外，您还可以使用 REST API 从外部托管的模型生成响应。那么，为什么要费力使用 Apache Kafka 呢？

有几个原因，例如：

  * **集成**: 许多企业希望运行自己的 LLM，以便将数据保留在内部。这需要将 LLM 驱动的组件集成到现有架构中，而这些架构可能已经使用某种消息总线进行了解耦。

  * **可扩展性**: Apache Kafka 在设计时就考虑了并行处理，因此许多团队倾向于使用它来更有效地将工作分配给可用的工作节点（在这种情况下，“工作节点”是运行 LLM 的容器）。

  * **持久性**: Kafka 的设计允许服务在另一个服务因内存问题或离线而中断后，能够接续其工作。这可以防止在多个系统相互通信的复杂分布式架构中（LLM 只是众多相互依赖的系统之一，还包括向量数据库和传统数据库）发生数据丢失。

有关事件流为何适合生成式 AI 应用程序架构的更多背景信息，请参阅 Kai Waehner 的文章 ["Apache Kafka + Vector Database + LLM = Real-Time GenAI"](https://www.kai-waehner.de/blog/2023/11/08/apache-kafka-flink-vector-database-llm-real-time-genai/)。